# Implementing Foundational ML Algorithms from scratch


----


## **Naive Bayes**

#### Introduction:

In this notebook, I’m extending my collection of machine learning algorithms implemented from scratch by adding the Naive Bayes classifier. A foundational probabilistic model that continues to deliver strong performance in text classification, spam filtering, and many other tasks.

Unlike models that rely on optimization techniques like Gradient Descent, Naive Bayes is built purely on probabilistic reasoning and statistical principles. Its simplicity, speed, and interpretability make it an essential addition to any machine learning toolbox.

#### Objectives:

- Implement a Binary Classification Model using Naive Bayes:

    1- Preprocess twitter messages labeled with positive and negative sentiment.

    2- Implement the Naive Bayes algorithm from scratch.

    3- Make predictions on labeled data.

    4- Evaluate its performance using accuracy as a metric.

-----


## 1- Preprocess twitter messages labeled with positive and negative sentiment

In [2]:
import nltk
from nltk.corpus import stopwords, twitter_samples

nltk.download('twitter_samples', download_dir='../data')
nltk.download('stopwords', download_dir='../data')

[nltk_data] Downloading package twitter_samples to ../data...
[nltk_data]   Unzipping corpora\twitter_samples.zip.
[nltk_data] Downloading package stopwords to ../data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [3]:
nltk.data.path.append('../data')


In [4]:
import numpy as np

Import the positive and negative tweets, then initialize the raw and test sets

In [5]:
all_positive_tweets = twitter_samples.strings('positive_tweets.json')
all_negative_tweets = twitter_samples.strings('negative_tweets.json')

# split the data
test_pos = all_positive_tweets[4000:]
train_pos = all_positive_tweets[:4000]
test_neg = all_negative_tweets[4000:]
train_neg = all_negative_tweets[:4000]

train_x = train_pos + train_neg
test_x = test_pos + test_neg

train_y = np.append(np.ones(len(train_pos)), np.zeros(len(train_neg)))
test_y = np.append(np.ones(len(test_pos)), np.zeros(len(test_neg)))

In [9]:
from pathlib import Path
import sys

# project root containing /tools (local tools and libs) and /notebooks
ROOT = Path.cwd().parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [10]:
from tools.utils_NLP import process_tweet, lookup

These two imported helper functions came from the NLP Specialization Course from Deeplearning.ai (Credits).

### `process_tweet` Function

This function takes a raw tweet as input and returns a cleaned, tokenized, and stemmed list of words. It is adapted from the DeepLearning.AI NLP Specialization course.

#### Input:
- `tweet`: A string containing a tweet.

#### Processing Steps:
1. Remove unwanted text patterns:
   - Stock tickers (e.g., `$GE`)
   - Retweet indicators (`RT`)
   - Hyperlinks (e.g., `http://...`)
   - Hashtag symbols (`#`, but keeps the word)

2. Tokenize the tweet:
   - Uses `TweetTokenizer` from NLTK.
   - Converts text to lowercase.
   - Removes Twitter handles.
   - Normalizes elongated words (e.g., "cooool" → "cool").

3. Clean tokens:
   - Removes stopwords using NLTK's English stopword list.
   - Removes punctuation.
   - Applies stemming using the Porter Stemmer.

#### Output:
- `tweets_clean`: A list of cleaned and stemmed words from the original tweet.

#### Example Use Case:
This function is useful in preprocessing text for NLP tasks such as sentiment analysis or topic classification.


In [11]:
custom_tweet = "RT @Twitter @chapagain Hello There! Have a great day. :) #good #morning http://chapagain.com.np"

# print cleaned tweet
print(process_tweet(custom_tweet))

['hello', 'great', 'day', ':)', 'good', 'morn']


### `count_tweets` Function

This function builds a frequency dictionary that maps each `(word, sentiment)` pair to the number of times it appears in a given list of tweets.

#### Inputs:
- `result`: A dictionary used to store the frequency of each `(word, sentiment)` pair.
- `tweets`: A list of tweet texts.
- `ys`: A list of sentiment labels (0 for negative, 1 for positive), corresponding to each tweet.

#### Process:
- For each tweet and its corresponding sentiment:
  - The tweet is preprocessed using the `process_tweet()` function.
  - Each word in the processed tweet is paired with the sentiment label.
  - The `(word, sentiment)` pair is added to the dictionary:
    - If it already exists, its count is incremented.
    - If it's new, it is added with a count of 1.

#### Output:
- Returns the updated `result` dictionary containing the frequency of each `(word, sentiment)` pair.
> Note: The `process_tweet()` function is expected to clean and tokenize the tweet (e.g., remove punctuation, lowercase, remove stopwords, etc.).


In [12]:
def count_tweets(result, tweets, ys):
    '''
    Input:
        result: a dictionary that will be used to map each pair to its frequency
        tweets: a list of tweets
        ys: a list corresponding to the sentiment of each tweet (either 0 or 1)
    Output:
        result: a dictionary mapping each pair to its frequency
    '''
    for y, tweet in zip(ys, tweets):
        for word in process_tweet(tweet):
            # define the key, which is the word and label tuple
            pair = (word,y)
            
            # if the key exists in the dictionary, increment the count
            if pair in result:
                result[pair] += 1

            # else, if the key is new, add it to the dictionary and set the count to 1
            else:
                result[pair] = 1
    
    return result

In [13]:
# Test
result = {}
tweets = ['i am happy', 'i am tricked', 'i am sad', 'i am tired', 'i am very tired']
ys = [1, 0, 0, 0, 0]
count_tweets(result, tweets, ys)

{('happi', 1): 1, ('trick', 0): 1, ('sad', 0): 1, ('tire', 0): 2}

----

## 2- Implement the Naive Bayes algorithm from scratch

Naive Bayes is a simple and fast algorithm commonly used for sentiment analysis. Training involves estimating the probability of each class and computing word likelihoods based on frequency.

### Class Probabilities (Priors)

We start by computing the prior probability of each class. The proportion of positive and negative tweets:

$$ P(D_{pos}) = \frac{D_{pos}}{D}, \quad P(D_{neg}) = \frac{D_{neg}}{D} $$

Where:
- $D$ is the total number of tweets
- $D_{pos}$ and $D_{neg}$ are the counts of positive and negative tweets respectively

The **logprior** helps simplify calculations:

$$ \text{logprior} = \log \left( \frac{P(D_{pos})}{P(D_{neg})} \right) = \log(D_{pos}) - \log(D_{neg}) $$

### Word Likelihoods

For each word in the vocabulary, we compute its likelihood given a class (positive or negative):

$$ P(w \mid pos) = \frac{freq_{pos} + 1}{N_{pos} + V} $$
$$ P(w \mid neg) = \frac{freq_{neg} + 1}{N_{neg} + V} $$

Where:
- `freq_pos`, `freq_neg`: how often the word appears in positive/negative tweets
- `N_pos`, `N_neg`: total word counts in positive/negative tweets
- `V`: vocabulary size (number of unique words)
- The `+1` is additive smoothing to handle unseen words




### Log Likelihood

To compare how strongly a word supports the positive vs. negative class, we compute its **log likelihood**:

$$ \text{loglikelihood} = \log \left( \frac{P(w \mid pos)}{P(w \mid neg)} \right) $$

This score reflects the strength and direction of association between a word and the sentiment class.

### Frequency Dictionary (`freqs`)

Before training, we build a frequency dictionary using the `count_tweets` function:

- The dictionary `freqs` maps `(word, label)` pairs to their count.
- It allows efficient lookup of how often each word appears in each class.
- We’ll use this dictionary throughout the model training process.

In [14]:
# Build the freqs dictionary
freqs = count_tweets({}, train_x, train_y)

In [15]:
def train_naive_bayes(freqs, train_x, train_y):
    '''
    Input:
        freqs: dictionary from (word, label) to how often the word appears
        train_x: a list of tweets
        train_y: a list of labels corresponding to the tweets (0,1)
    Output:
        logprior: the log prior. (equation 3 above)
        loglikelihood: the log likelihood of you Naive bayes equation. (equation 6 above)
    '''
    loglikelihood = {}
    logprior = 0

    # calculate V, the number of unique words in the vocabulary
    vocab = list(set([key[0] for key in freqs.keys()]))
    V = len(vocab)

    # calculate N_pos, N_neg
    N_pos = N_neg = 0
    for pair in freqs.keys():
        
        if pair[1] > 0:
            N_pos += freqs[pair]
        else:
            N_neg += freqs[pair]
            
    # Calculate D, the number of documents
    D = len(train_x)

    # Calculate D_pos, the number of positive documents
    D_pos = sum(train_y)

    # Calculate D_neg, the number of negative documents
    D_neg = D - D_pos

    # Calculate logprior
    logprior = np.log(D_pos) - np.log(D_neg)
    
    # For each word in the vocabulary...
    for word in vocab:
        # get the positive and negative frequency of the word
        freq_pos = freqs.get((word,1),0)
        freq_neg = freqs.get((word,0),0)

        # calculate the probability that each word is positive, and negative
        p_w_pos = (freq_pos + 1)/(N_pos + V)
        p_w_neg = (freq_neg + 1)/(N_neg + V)

        # calculate the log likelihood of the word
        loglikelihood[word] = np.log(p_w_pos) - np.log(p_w_neg)

    return logprior, loglikelihood

In [16]:
logprior, loglikelihood = train_naive_bayes(freqs, train_x, train_y)
print(logprior)
print(len(loglikelihood))

0.0
9143


### Training Output Summary
After training the Naive Bayes model:
- **logprior = 0.0**: This indicates that the number of positive and negative tweets in the training set is equal, resulting in a neutral prior (i.e., no class is favored a priori).
- **loglikelihood contains 9,143 words**: This is the size of the vocabulary learned from the training data. Each word has an associated log likelihood score indicating how strongly it supports a positive vs. negative sentiment.

These values will be used during prediction to score new tweets based on the sum of their word log likelihoods and the logprior.


----

## 3- Make predictions on labeled data.

With the `logprior` and `loglikelihood` computed during training, we can now test the model by making predictions on new tweets.

### Task: `naive_bayes_predict`

Implement a function that predicts the sentiment of a tweet using:

$$ p = \text{logprior} + \sum_{i=1}^{N} \text{loglikelihood}_i $$

**Instructions:**
- Input: a tweet, the `logprior`, and the `loglikelihood` dictionary
- For each word in the processed tweet:
  - If it exists in `loglikelihood`, add its score to the total
- Add the `logprior` to the final sum
- Output: the overall sentiment score (positive if > 0, negative if < 0)

### Note

In our training set, classes are balanced (4000 positive and 4000 negative tweets), so:

- The prior ratio is 1 → `logprior = 0.0`
- In this case, the prediction depends only on the loglikelihoods.
- Still, always include `logprior`, as it becomes important when data is imbalanced.


In [17]:
def naive_bayes_predict(tweet, logprior, loglikelihood):
    '''
    Input:
        tweet: a string
        logprior: a number
        loglikelihood: a dictionary of words mapping to numbers
    Output:
        p: the sum of all the logliklihoods of each word in the tweet (if found in the dictionary) + logprior (a number)

    '''
    # process the tweet to get a list of words
    word_l = process_tweet(tweet)

    # initialize probability to zero
    p = 0

    # add the logprior
    p += logprior

    for word in word_l:
        # check if the word exists in the loglikelihood dictionary
        if word in loglikelihood:
            # add the log likelihood of that word to the probability
            p += loglikelihood[word]

    return p

In [18]:
# Test 1
my_tweet = 'She smiled :)'
p = naive_bayes_predict(my_tweet, logprior, loglikelihood)
print('The expected output is', p)

The expected output is 8.434581042573853


----

## 4- Evaluate the performance using accuracy as a metric

In [19]:
def test_naive_bayes(test_x, test_y, logprior, loglikelihood, naive_bayes_predict=naive_bayes_predict):
    """
    Input:
        test_x: A list of tweets
        test_y: the corresponding labels for the list of tweets
        logprior: the logprior
        loglikelihood: a dictionary with the loglikelihoods for each word
    Output:
        accuracy: (# of tweets classified correctly)/(total # of tweets)
    """
    accuracy = 0  # return this properly

    y_hats = []
    for tweet in test_x:
        if naive_bayes_predict(tweet, logprior, loglikelihood) > 0:
            y_hat_i = 1
        else:
            y_hat_i = 0
        y_hats.append(y_hat_i)

    # error is the average of the absolute values of the differences between y_hats and test_y
    error = np.mean(np.abs(y_hats - test_y))

    accuracy = 1 - error

    return accuracy

In [20]:
print("Naive Bayes accuracy = %0.4f" %
      (test_naive_bayes(test_x, test_y, logprior, loglikelihood)))

Naive Bayes accuracy = 0.9955


### Model Evaluation Result

The Naive Bayes classifier achieved an accuracy of **0.9955** on the test set.

This means the model correctly classified ~99.5% of the tweets, which is an excellent result, especially given that the implementation is from scratch and based on simple statistical assumptions.

> Keep in mind that such high accuracy could be influenced by the balance and simplicity of the dataset (preprocessed Twitter sentiment samples). For real-world applications, further validation (e.g., using precision, recall, and testing on more diverse data) would be recommended.


In [21]:
# Test the function
for tweet in ['I am happy', 'I am bad', 'this movie should have been great.', 'great', 'great great', 'great great great', 'great great great great']:
    p = naive_bayes_predict(tweet, logprior, loglikelihood)
    print(f'{tweet} -> {p:.2f}')

I am happy -> 2.13
I am bad -> -1.31
this movie should have been great. -> 2.11
great -> 2.13
great great -> 4.25
great great great -> 6.38
great great great great -> 8.50


## 🏁 Conclusion

In this notebook, we successfully implemented the **Naive Bayes classifier from scratch** for binary sentiment analysis on Twitter data.

### Key Takeaways:
- We used NLTK's `twitter_samples` and `stopwords` corpora to build and clean the dataset.
- A custom `process_tweet` function helped normalize and tokenize the tweets for modeling.
- We built a `count_tweets` frequency dictionary to capture the relationship between words and sentiment labels.
- The Naive Bayes model was trained using log priors and log likelihoods, enabling efficient probabilistic predictions.
- Our classifier achieved an impressive **accuracy of 0.9955** on the test set, correctly classifying ~99.5% of the tweets.

### Final Test Observations:
- The predictions aligned well with expectations, positive phrases (e.g., `"I am happy"` or repeated `"great"`) produced strong positive scores.
- Negative sentiment (e.g., `"I am bad"`) resulted in negative scores as expected.
- Repetition of a positively weighted word increased the prediction score linearly, demonstrating the additive nature of log likelihoods.

> While the model performed exceptionally on this dataset, real-world applications may require additional validation using more diverse and imbalanced data.

This project demonstrates how probabilistic models like Naive Bayes can be both **interpretable and effective**, especially when built with a deep understanding of the underlying mechanics.



### Limitations of Naive Bayes

Despite **its simplicity and strong baseline performance**, Naive Bayes comes with important limitations:

- **Independence Assumption**: It assumes that words are conditionally independent given the class label, which is rarely true in natural language (e.g., "not good" vs. "good").
- **Lack of Context**: The model doesn't capture the order or context of words, meaning it treats all words equally regardless of their position or relationship.
- **Vulnerability to Adversarial Inputs**: Because it's based on word counts and additive scoring, it's easy to manipulate predictions by repeating highly weighted words.

These factors make Naive Bayes less suitable for tasks requiring deeper language understanding, where more sophisticated models may perform better.


----